In [2]:
!pip install rapidfuzz

In [3]:
import pandas as pd
import joblib

from rapidfuzz import process, fuzz

In [12]:
movies = pd.read_pickle("../models/movies.pkl")

movies.head()

,movie_id,title,release_year,poster_url,avg_rating,genres,languages,countries,director,main_cast,overview,tags
0,1000001,Barbie,2023.0,https://a.ltrbxd.com/resized/film-poster/2/7/7...,3.86,comedy adventure,english,uk usa,greta gerwig,margot robbie as barbie ryan gosling as ken am...,barbie and ken are having the time of their li...,comedy adventure comedy adventure comedy adven...
1,1000002,Parasite,2019.0,https://a.ltrbxd.com/resized/film-poster/4/2/6...,4.56,comedy thriller drama,korean english german,south korea,bong joon ho,song kang ho as kim ki taek lee sun kyun as pa...,all unemployed ki taek s family takes peculiar...,comedy thriller drama comedy thriller drama co...
2,1000003,Everything Everywhere All at Once,2022.0,https://a.ltrbxd.com/resized/film-poster/4/7/4...,4.30,science fiction adventure comedy action,english cantonese chinese,usa,daniel scheinert daniel kwan,michelle yeoh as evelyn wang ke huy quan as wa...,an aging chinese immigrant is swept up in an i...,science fiction adventure comedy action scienc...
3,1000004,Fight Club,1999.0,https://a.ltrbxd.com/resized/film-poster/5/1/5...,4.27,drama,english,germany usa,david fincher,edward norton as narrator brad pitt as tyler d...,a ticking time bomb insomniac and a slippery s...,drama drama drama david fincher david fincher ...
4,1000005,La La Land,2016.0,https://a.ltrbxd.com/resized/film-poster/2/4/0...,4.09,drama comedy music romance,english,hong kong usa,damien chazelle,ryan gosling as sebastian emma stone as mia jo...,mia an aspiring actress serves lattes to movie...,drama comedy music romance drama comedy music ...


In [13]:
movies["display_title"] = (
    movies["title"] +
    " (" +
    movies["release_year"].fillna(0).astype(int).astype(str) +
    ")"
)

In [14]:
movie_titles = movies["display_title"].tolist()

In [19]:
def prefix_search(query):

    query = query.lower()

    return movies[
        movies["display_title"]
        .str.lower()
        .str.startswith(query)
    ]

In [21]:
prefix_search("inter")

,movie_id,title,release_year,poster_url,avg_rating,genres,languages,countries,director,main_cast,overview,tags,display_title
6,1000007,Interstellar,2014.0,https://a.ltrbxd.com/resized/film-poster/1/1/7...,4.35,science fiction drama adventure,english,uk usa,christopher nolan,matthew mcconaughey as joseph coop cooper anne...,the adventures of a group of explorers who mak...,science fiction drama adventure science fictio...,Interstellar (2014)
822,1000830,Interview with the Vampire,1994.0,https://a.ltrbxd.com/resized/film-poster/5/1/4...,3.58,horror drama fantasy,english french,usa,neil jordan,tom cruise as lestat brad pitt as louis antoni...,a vampire relates his epic life story of love ...,horror drama fantasy horror drama fantasy horr...,Interview with the Vampire (1994)
4698,1004827,Interstella 5555: The 5tory of the 5ecret 5tar...,2003.0,https://a.ltrbxd.com/resized/film-poster/4/5/6...,4.14,adventure science fiction animation music,english,france japan,kazuhisa takenouchi daisuke nishio bernard dey...,romanthony as octave singing voice thomas bang...,four talented alien musicians are kidnapped by...,adventure science fiction animation music adve...,Interstella 5555: The 5tory of the 5ecret 5tar...
7111,1007422,Interiors,1978.0,https://a.ltrbxd.com/resized/film-poster/4/2/0...,3.72,drama,english,usa,woody allen,geraldine page as eve kristin griffith as flyn...,when eve an interior designer is deserted by h...,drama drama drama woody allen woody allen wood...,Interiors (1978)
9948,1010521,Internal Affairs,1990.0,https://a.ltrbxd.com/resized/film-poster/4/5/6...,3.21,drama crime,english spanish,canada usa,mike figgis,richard gere as dennis peck andy garc a as ray...,keen young raymold avila joins the internal af...,drama crime drama crime drama crime mike figgi...,Internal Affairs (1990)
...,...,...,...,...,...,...,...,...,...,...,...,...,...
559629,1918544,Interface,2009.0,NaN,NaN,thriller drama,english,usa,sahand nikoukar,nick medina as rob eric adrian marshall as tre...,a wandering computer programmer befriends coll...,thriller drama thriller drama thriller drama s...,Interface (2009)
564974,1926581,Interloop,NaN,https://a.ltrbxd.com/resized/film-poster/9/5/3...,NaN,action mystery science fiction,english,usa,sarbesh rajarajan,arish vadnere as john hrishi kamal as amon swi...,a normal guy goes on about his daily life when...,action mystery science fiction action mystery ...,Interloop (0)
564975,1926582,Internal Enemy,NaN,NaN,NaN,documentary,portuguese,brazil,camilo tavares,cast information unavailable,this investigative documentary uses confidenti...,documentary documentary documentary camilo tav...,Internal Enemy (0)
564976,1926586,Interview with a Homicide Detective,NaN,NaN,NaN,NaN,english,unknown,NaN,dian van patten as thelma james e welsh as det...,a grizzled bettered detective makes a comeback...,dian van patten as thelma james e welsh as det...,Interview with a Homicide Detective (0)


In [36]:

def search_movie(query, limit=10):

    # -----------------------------
    # Input Validation
    # -----------------------------
    query = query.strip()

    if query == "":
        return pd.DataFrame()

    query_lower = query.lower()

    limit = max(1, min(limit, 50))

    # -----------------------------
    # Prefix Search
    # -----------------------------
    prefix = movies[
        movies["display_title"]
        .str.lower()
        .str.startswith(query_lower, na=False)
    ].copy()

    prefix["search_score"] = 100

    # -----------------------------
    # Contains Search
    # -----------------------------
    contains = movies[
        movies["display_title"]
        .str.lower()
        .str.contains(query_lower, na=False)
    ].copy()

    contains = contains[
        ~contains.index.isin(prefix.index)
    ]

    contains["search_score"] = 95

    combined = pd.concat([prefix, contains])

    # -----------------------------
    # If enough results found
    # -----------------------------
    if len(combined) >= limit:

        combined = combined.sort_values(
            by=["search_score", "avg_rating"],
            ascending=[False, False],
            na_position="last"
        )

        return combined[
            [
                "movie_id",
                "title",
                "display_title",
                "release_year",
                "avg_rating",
                "poster_url",
                "search_score"
            ]
        ].head(limit)

    # -----------------------------
    # RapidFuzz Search
    # -----------------------------
    fuzzy = process.extract(
        query,
        movies["display_title"],
        scorer=fuzz.WRatio,
        limit=limit * 3
    )

    fuzzy = [
        item
        for item in fuzzy
        if item[1] >= 70
    ]

    fuzzy_rows = []

    for title, score, idx in fuzzy:

        if idx in combined.index:
            continue

        row = movies.iloc[idx].copy()

        row["search_score"] = score

        fuzzy_rows.append(row)

    if len(fuzzy_rows):

        fuzzy_rows = pd.DataFrame(fuzzy_rows)

        final = pd.concat([combined, fuzzy_rows])

    else:

        final = combined

    # -----------------------------
    # Remove Duplicates
    # -----------------------------
    final = final.drop_duplicates(
        subset="movie_id"
    )

    # -----------------------------
    # Sort Results
    # -----------------------------
    final = final.sort_values(
        by=["search_score", "avg_rating"],
        ascending=[False, False],
        na_position="last"
    )

    # -----------------------------
    # Return
    # -----------------------------
    return final[
        [
            "movie_id",
            "title",
            "display_title",
            "release_year",
            "avg_rating",
            "poster_url",
            "search_score"
        ]
    ].head(limit)



In [37]:
search_movie("inter")

,movie_id,title,display_title,release_year,avg_rating,poster_url,search_score
6,1000007,Interstellar,Interstellar (2014),2014.0,4.35,https://a.ltrbxd.com/resized/film-poster/1/1/7...,100
17872,1019391,Interface,Interface (2022),2022.0,4.15,https://a.ltrbxd.com/resized/film-poster/9/4/4...,100
4698,1004827,Interstella 5555: The 5tory of the 5ecret 5tar...,Interstella 5555: The 5tory of the 5ecret 5tar...,2003.0,4.14,https://a.ltrbxd.com/resized/film-poster/4/5/6...,100
14418,1015494,Interrogation,Interrogation (1989),1989.0,4.03,https://a.ltrbxd.com/resized/film-poster/4/0/2...,100
21461,1023508,Interview,Interview (1971),1971.0,3.81,https://a.ltrbxd.com/resized/film-poster/2/0/9...,100
29253,1032611,Intermissions,Intermissions (2004),2004.0,3.80,https://a.ltrbxd.com/resized/film-poster/1/7/6...,100
7111,1007422,Interiors,Interiors (1978),1978.0,3.72,https://a.ltrbxd.com/resized/film-poster/4/2/0...,100
52979,1062301,Interactive Introverts,Interactive Introverts (2018),2018.0,3.70,https://a.ltrbxd.com/resized/film-poster/5/0/1...,100
15209,1016380,Intervista,Intervista (1987),1987.0,3.59,https://a.ltrbxd.com/resized/film-poster/1/7/7...,100
822,1000830,Interview with the Vampire,Interview with the Vampire (1994),1994.0,3.58,https://a.ltrbxd.com/resized/film-poster/5/1/4...,100


In [38]:
search_movie("bat")

,movie_id,title,display_title,release_year,avg_rating,poster_url,search_score
12037,1012819,Battlestar Galactica,Battlestar Galactica (2003),2003.0,4.10,https://a.ltrbxd.com/resized/film-poster/2/5/2...,100
2005,1002041,Batman: Mask of the Phantasm,Batman: Mask of the Phantasm (1993),1993.0,4.01,https://a.ltrbxd.com/resized/film-poster/4/2/7...,100
2107,1002147,Battleship Potemkin,Battleship Potemkin (1925),1925.0,4.00,https://a.ltrbxd.com/resized/film-poster/5/1/4...,100
3833,1003924,"Batman: The Dark Knight Returns, Part 2","Batman: The Dark Knight Returns, Part 2 (2013)",2013.0,3.94,https://a.ltrbxd.com/resized/sm/upload/nx/8p/3...,100
2868,1002931,Batman: Under the Red Hood,Batman: Under the Red Hood (2010),2010.0,3.90,https://a.ltrbxd.com/resized/film-poster/2/4/3...,100
935,1000944,Battle Royale,Battle Royale (2000),2000.0,3.86,https://a.ltrbxd.com/resized/film-poster/4/9/8...,100
3656,1003743,"Batman: The Dark Knight Returns, Part 1","Batman: The Dark Knight Returns, Part 1 (2012)",2012.0,3.81,https://a.ltrbxd.com/resized/sm/upload/fj/e7/j...,100
7848,1008231,Battles Without Honor and Humanity,Battles Without Honor and Humanity (1973),1973.0,3.81,https://a.ltrbxd.com/resized/film-poster/2/9/2...,100
169,1000170,Batman Begins,Batman Begins (2005),2005.0,3.80,https://a.ltrbxd.com/resized/film-poster/5/1/7...,100
13547,1014533,Battles Without Honor and Humanity: Deadly Fig...,Battles Without Honor and Humanity: Deadly Fig...,1973.0,3.79,https://a.ltrbxd.com/resized/film-poster/1/2/9...,100


In [39]:
search_movie("dark")

,movie_id,title,display_title,release_year,avg_rating,poster_url,search_score
10069,1010654,Dark Days,Dark Days (2000),2000.0,4.01,https://a.ltrbxd.com/resized/film-poster/4/3/2...,100
1841,1001871,Dark Waters,Dark Waters (2019),2019.0,3.76,https://a.ltrbxd.com/resized/film-poster/4/8/0...,100
6679,1006940,Dark Passage,Dark Passage (1947),1947.0,3.72,https://a.ltrbxd.com/resized/film-poster/4/1/7...,100
2159,1002201,Dark City,Dark City (1998),1998.0,3.69,https://a.ltrbxd.com/resized/film-poster/5/0/1...,100
10755,1011392,Dark Victory,Dark Victory (1939),1939.0,3.64,https://a.ltrbxd.com/resized/film-poster/3/4/0...,100
46723,1054258,Darkness and Light,Darkness and Light (1999),1999.0,3.64,https://a.ltrbxd.com/resized/film-poster/1/2/9...,100
38139,1043461,Dark River,Dark River (1952),1952.0,3.62,https://a.ltrbxd.com/resized/film-poster/2/4/5...,100
60273,1071995,Dark Suns,Dark Suns (2018),2018.0,3.61,https://a.ltrbxd.com/resized/film-poster/4/8/8...,100
23340,1025680,Dark Eyes,Dark Eyes (1987),1987.0,3.60,https://a.ltrbxd.com/resized/film-poster/2/0/5...,100
63801,1076658,Dark Wind,Dark Wind (2017),2017.0,3.57,https://a.ltrbxd.com/resized/film-poster/4/1/9...,100


In [40]:
search_movie("parasit")

,movie_id,title,display_title,release_year,avg_rating,poster_url,search_score
1,1000002,Parasite,Parasite (2019),2019.0,4.56,https://a.ltrbxd.com/resized/film-poster/4/2/6...,100
25263,1027923,Parasite Dolls,Parasite Dolls (2003),2003.0,3.26,https://a.ltrbxd.com/resized/film-poster/9/9/7...,100
30715,1034422,Parasite Eve,Parasite Eve (1997),1997.0,3.18,https://a.ltrbxd.com/resized/film-poster/3/0/0...,100
15331,1016518,Parasite in Love,Parasite in Love (2021),2021.0,3.10,https://a.ltrbxd.com/resized/film-poster/6/3/2...,100
71587,1087329,Parasites,Parasites (2016),2016.0,3.09,https://a.ltrbxd.com/resized/film-poster/3/3/7...,100
19177,1020865,Parasite,Parasite (1982),1982.0,2.38,https://a.ltrbxd.com/resized/film-poster/1/7/2...,100
93910,1118497,Parasite,Parasite (2004),2004.0,NaN,https://a.ltrbxd.com/resized/film-poster/2/1/7...,100
110069,1141861,Parasitic,Parasitic (2012),2012.0,NaN,https://a.ltrbxd.com/resized/film-poster/1/6/7...,100
141661,1189184,Parasite,Parasite (2014),2014.0,NaN,https://a.ltrbxd.com/resized/film-poster/2/2/2...,100
264168,1395935,Parasite Challenge Double-Up WJSN,Parasite Challenge Double-Up WJSN (2022),2022.0,NaN,https://a.ltrbxd.com/resized/film-poster/9/4/0...,100


In [41]:
search_movie("xyzabc")

,movie_id,title,display_title,release_year,avg_rating,poster_url,search_score


In [42]:
from collections import defaultdict

prefix_index = defaultdict(list)

for idx, title in enumerate(movies["display_title"]):

    if pd.isna(title):
        continue

    title = title.lower()

    words = title.split()

    for word in words:

        for i in range(1, min(len(word), 6) + 1):

            prefix = word[:i]

            prefix_index[prefix].append(idx)

print("Prefix Index Created")
print(f"Total Prefixes : {len(prefix_index):,}")

Prefix Index Created
Total Prefixes : 363,885


In [43]:
import joblib

joblib.dump(
    prefix_index,
    "../models/prefix_index.pkl"
)

print("Prefix Index Saved")

Prefix Index Saved


In [44]:
def fast_search(query):

    query = query.lower().strip()

    return prefix_index.get(query, [])

In [46]:
import time

start = time.time()

fast_search("inter")

print(time.time() - start)

0.00020551681518554688
